# Otimização — capítulo 6: Otimização multiobjetivo
### Exemplos em Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/optimization-course/blob/main/code/python/notebooks/06_multiobjective_optimization.ipynb)

Cada secção corre o exemplo `ex….py` do método e compara os resultados com os slides (no fim: *confere com os slides: sim*). Comece pela célula **Preparação**; no Colab, *Runtime → Run all* corre tudo.

Os slides usam vírgula decimal; aqui usa-se o ponto. Este caderno é gerado a partir dos ficheiros `ex….py` por `make_notebooks.py`.

*Examples for Chapter 6 of the course "Optimization" (ISEL, J. F. A. Madeira). Code comments and printed messages are in Portuguese; the English slides are in [`notes/en/`](https://github.com/aguilarmadeira/optimization-course/tree/main/notes/en).*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
if os.path.isfile(os.path.join("..", "uc_setup.py")):
    RAIZ = os.path.abspath("..")
else:
    if not os.path.isdir("optimization-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/optimization-course
    RAIZ = os.path.abspath(os.path.join("optimization-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 6.3 Métodos de agregação de objetivos

Exemplo [`ex06_3_aggregation.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/06_multiobjective_optimization/06_3_aggregation/ex06_3_aggregation.py) · funções da UC: [`aggregation.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/06_multiobjective_optimization/06_3_aggregation/aggregation.py)

```text
Reproduz os exemplos do deck 6.3 (agregação de objetivos).

Exemplo convexo da UC (6.1):  f1 = x1^2 + x2^2, f2 = (x1 - 1)^2 + x2^2,
x em [0,1]^2, x1^2 + x2^2 <= 1.
  1. «O que fazem os pesos?» e «Pesos uniformes ≠ pontos uniformes»: soma
     ponderada com 11 pesos w1 = 0, 0.1, ..., 1, x0 = (0.5, 0.3), arranque a
     quente, SLSQP: x1* = w2, f1 = w2^2, f2 = (1 - w2)^2 (tabela); n_f = 75.
Exemplo não convexo da UC (6.2):  f1 = x1, f2 = 1 - x1^2 + x2,
x em [0,1] x [0, 0.6].
  2. «A limitação fundamental»: com w1, w2 > 0 a soma ponderada só dá os dois
     extremos (supported): 11, 100 e 10 000 pesos -> 2 pontos.
  3. «Tchebycheff ponderado»: min theta s.a. w_i (f_i - z_i^id) <= theta,
     z^id = (0, 0): encontra também os pontos non-supported.
4. «A escala importa»: projetos A, B, C; w = (0.9, 0.1) sem normalizar dá
   81.4 / 85.8 / 100.5 (ganha A); normalizado com ideal e nadir, C / B / A.

Usa aggregation.py (nesta pasta). O n_f = 75 é o do SLSQP do SciPy (como no
script das figuras) e verifica-se aqui, não no MATLAB/Octave (outro solver).
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [2]:
import numpy as np

from aggregation import weighted_sum, weighted_tchebycheff


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def perto(v, s, tol):
    """v está a menos de tol do valor exato s?"""
    return bool(np.all(np.abs(np.ravel(np.asarray(v, float))
                              - np.ravel(np.asarray(s, float))) <= tol))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


limpa = lambda v: np.where(np.abs(v) >= 5e-5, v, 0.0) + 0.0     # não imprimir -0.0000
ok = True
print("Otimização — deck 6.3: métodos de agregação de objetivos")

# ------------------------------------------------ 1. Exemplo convexo
F = lambda x: np.array([x[0]**2 + x[1]**2, (x[0] - 1)**2 + x[1]**2])
cons = [{"type": "ineq", "fun": lambda x: 1 - x[0]**2 - x[1]**2}]   # forma da UC: g >= 0
b = [(0, 1), (0, 1)]
x0 = [0.5, 0.3]
w1 = np.linspace(0, 1, 11); W = np.c_[w1, 1 - w1]
print("\n1. Exemplo convexo: min w1 f1 + w2 f2, 11 pesos, x0 = (%g, %g), arranque a quente" % tuple(x0))
r = weighted_sum(F, x0, b, W, cons, verbose=True)
print("   solver: %s" % r.message)
print("   n_f = %d (soma de r.nfev; inclui as avaliações dos gradientes por diferenças finitas)" % r.nfev)
T = [[0.0, 1.0, 1.000, 1.000, 0.000], [0.1, 0.9, 0.900, 0.810, 0.010], [0.2, 0.8, 0.800, 0.640, 0.040],
     [0.3, 0.7, 0.700, 0.490, 0.090], [0.4, 0.6, 0.600, 0.360, 0.160], [0.5, 0.5, 0.500, 0.250, 0.250],
     [0.6, 0.4, 0.400, 0.160, 0.360], [0.7, 0.3, 0.300, 0.090, 0.490], [0.8, 0.2, 0.200, 0.040, 0.640],
     [0.9, 0.1, 0.100, 0.010, 0.810], [1.0, 0.0, 0.000, 0.000, 1.000]]
c = [confere(np.c_[W, r.x[:, 0], r.fx], T, 3), perto(r.x[:, 0], W[:, 1], 1e-5), perto(r.x[:, 1], 0, 1e-5),
     confere(r.x[[10, 5, 0]], [[0, 0], [0.5, 0], [1, 0]], 3), r.flag == 0, r.nfev == 75]
print("  tabela (11 pesos): %s | x1* = w2: %s | x2* = 0: %s | w = (1;0), (0.5;0.5), (0;1) -> (0;0), (0.5;0), (1;0): %s"
      " | sem falhas: %s | n_f = 75: %s" % simnao(c))
ok = ok and all(c)
d = np.diff(r.fx[:, 0])
print("   pesos uniformes, pontos não uniformes: passos em f1 de %.2f (junto a f1 = 1) a %.2f (junto a f1 = 0)"
      % (-d[0], -d[-1]))

# ------------------------------------------------ 2. Exemplo não convexo
F = lambda x: np.array([x[0], 1 - x[0]**2 + x[1]])
b = [(0, 1), (0, 0.6)]
X0 = [[0.1, 0.1], [0.5, 0.1], [0.9, 0.1]]          # três pontos iniciais (solver local)
print("\n2. Exemplo não convexo: soma ponderada, 3 pontos iniciais por peso, fica o melhor")
w1 = np.array([0.2, 0.5, 0.8])
r = weighted_sum(F, X0, b, np.c_[w1, 1 - w1])
for wk, xk, Fk in zip(w1, r.x, r.fx):
    print("   w1 = %.1f: x* = (%.4f; %.4f), (f1, f2) = (%.4f; %.4f)" % (wk, *limpa(np.r_[xk, Fk])))
ra = weighted_sum(F, [0.5, 0.1], b, [[0.5, 0.5]])
xa = ra.x[0]
print("   w1 = 0.5 só a partir de (0.5; 0.1): x = (%.4f; %.4f), f~ = %.4f"
      % (*limpa(xa), 0.5 * ra.fx[0].sum()), end="")
if 0.01 < xa[0] < 0.99:
    print("\n   (cuidado: ponto estacionário de f~, côncava em x1 — é um máximo, não um mínimo)")
else:
    print(" (um extremo)")
c = []
for N in (11, 100, 10000):
    w1 = np.linspace(0, 1, N)
    pos = (w1 > 0) & (w1 < 1)                       # w1, w2 > 0
    if N < 10000:
        Fx = weighted_sum(F, X0, b, np.c_[w1, 1 - w1]).fx
        como = "solver"
    else:                                           # 10 000 pesos: pesquisa exaustiva em x1
        t = np.linspace(0, 1, 1001)                 # (com w2 > 0 o ótimo tem x2 = 0)
        wp = w1[pos][:, None]
        j = np.argmin(wp * t + (1 - wp) * (1 - t**2), axis=1)
        Fx = np.zeros((N, 2)); Fx[pos] = np.c_[t[j], 1 - t[j]**2]
        como = "grelha"
    P = np.unique(np.round(Fx[pos] * 1e6) / 1e6, axis=0) + 0.0
    print("   %5d pesos (%s), %5d com w1, w2 > 0 -> %d pontos distintos: %s"
          % (N, como, pos.sum(), len(P), " ".join("(%g; %g)" % tuple(p) for p in P)))
    c.append(len(P) == 2 and perto(P[np.lexsort(P.T[::-1])], [[0, 1], [1, 0]], 1e-6))
print("   (com w1 = 1, w2 = 0, x2 fica livre: o solver devolve (0; 1.1), só fracamente eficiente)")
print("  11 pesos -> 2 pontos: %s | 100 -> 2: %s | 10 000 -> 2: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 3. Tchebycheff ponderado
zid = np.array([0.0, 0.0])
w1 = np.linspace(0, 1, 11); W = np.maximum(np.c_[w1, 1 - w1], 1e-3)   # como no script das figuras
print("\n3. Tchebycheff ponderado (min theta), z^id = (0, 0), 11 pesos (w_i >= 1e-3), a quente")
r = weighted_tchebycheff(F, [0.5, 0.3], b, W, zid, verbose=True)
print("   solver: %s; n_f = %d chamadas a F" % (r.message, r.nfev))
a, bb = W[:, 0], W[:, 1]                             # exato: w1 f1 = w2 (1 - f1^2)
f1ex = (-a + np.sqrt(a**2 + 4 * bb**2)) / (2 * bb)
nint = int(np.sum((r.fx[:, 0] > 0.01) & (r.fx[:, 0] < 0.99)))
print("   f1 exato (w1 f1 = w2 (1 - f1^2)):" + "".join(" %.4f" % v for v in f1ex))
print("   %d dos 11 pontos são interiores (non-supported)" % nint)
c = [perto(r.fx[:, 0], f1ex, 1e-3), perto(r.fx[:, 1], 1 - f1ex**2, 2e-3), nint == 9, r.flag == 0]
print("  pontos na frente (cruzamento w1 f1 = w2 f2): %s | f2 = 1 - f1^2: %s | 9 non-supported: %s"
      " | sem falhas: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 4. A escala importa
print("\n4. A escala importa: f1 = massa (kg), f2 = custo (€)")
Y = np.array([[1.5, 800], [0.9, 850], [0.5, 1000]]); nomes = "ABC"
s = Y @ [0.9, 0.1]; j = int(np.argmin(s))
print("   w = (0.9; 0.1), sem normalizar: A = %.2f, B = %.2f, C = %.2f -> ganha %s" % (*s, nomes[j]))
zi, zn = Y.min(0), Y.max(0)                          # os três são eficientes
Yn = (Y - zi) / (zn - zi)
Wn = np.array([[0.9, 0.1], [0.5, 0.5], [0.1, 0.9]])
print("   normalizado com z^id = (%g; %g), z^nad = (%g; %g):" % (*zi, *zn))
for w in Wn:
    print("     w = (%.1f; %.1f): A = %.3f, B = %.3f, C = %.3f" % (*w, *(Yn @ w)))
esc = "".join(nomes[i] for i in np.argmin(Yn @ Wn.T, axis=0))
print("     escolhe %s" % " / ".join(esc))
c = [confere(s, [81.4, 85.8, 100.5], 1), j == 0, esc == "CBA"]
print("  81.4 / 85.8 / 100.5: %s | ganha A: %s | normalizado C / B / A: %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 6.3: métodos de agregação de objetivos

1. Exemplo convexo: min w1 f1 + w2 f2, 11 pesos, x0 = (0.5, 0.3), arranque a quente
      w1      w2      x1      x2      f1      f2      nf
    0.00    1.00  1.0000  0.0000  1.0000  0.0000       6
    0.10    0.90  0.9000  0.0000  0.8100  0.0100       7
    0.20    0.80  0.8000  0.0000  0.6400  0.0400       7
    0.30    0.70  0.7000  0.0000  0.4900  0.0900       7
    0.40    0.60  0.6000  0.0000  0.3600  0.1600       7
    0.50    0.50  0.5000  0.0000  0.2500  0.2500       7
    0.60    0.40  0.4000  0.0000  0.1600  0.3600       7
    0.70    0.30  0.3000  0.0000  0.0900  0.4900       7
    0.80    0.20  0.2000  0.0000  0.0400  0.6400       7
    0.90    0.10  0.1000  0.0000  0.0100  0.8100       7
    1.00    0.00  0.0000  0.0000  0.0000  1.0000       6
   solver: SLSQP: 11 problemas escalares resolvidos
   n_f = 75 (soma de r.nfev; inclui as avaliações dos gradientes por diferenças finitas)
  tabela (11 pesos): sim | x1* = 

## 6.4 Método do ε-constrangimento

Exemplo [`ex06_4_epsilon_constraint.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/06_multiobjective_optimization/06_4_epsilon_constraint/ex06_4_epsilon_constraint.py) · funções da UC: [`epsilon_constraint.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/06_multiobjective_optimization/06_4_epsilon_constraint/epsilon_constraint.py)

```text
Reproduz os exemplos do deck 6.4 (epsilon-constrangimento).

Exemplo não convexo da UC (6.2):  min (f1, f2) = (x1, 1 - x1^2 + x2),
x em [0,1] x [0, 0.6].  Subproblema:  min f2  s.a.  f1 <= e.
  1. «O mesmo exemplo do 6.3, à mão»: e = 0.25, 0.50, 0.75.
  2. «Em MATLAB: varrer e com fmincon» / «O mesmo em Python»: 10 valores
     e = 0.05, 0.15, ..., 0.95, x0 = (0.5, 0.3), arranque a quente, SLSQP:
     10 pontos, 60 avaliações de f2, 90 de f1 (150 chamadas), u = 2e.
  3. «O multiplicador: a taxa de troca»: e = 0.5 -> u = 1; f2(0.51) = 0.7399;
     e = 0.9 -> u = 1.8, a mesma folga compra ~0.018.
  4. «Restrição inativa»: e = 1.05 e 1.2 -> a mesma solução (1, 0), u = 0.
  5. «Nantes–Lille»: o mais barato com duração <= e (e = 4, 5, 10 h).

As contagens 60/90/150 são as do SLSQP do SciPy (como no script das figuras,
com a restrição na forma da UC, {'type': 'ineq', 'fun': e - f1}); verificam-se
aqui e não no MATLAB/Octave (outro solver).
Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [3]:
import warnings

import numpy as np
from scipy.optimize import NonlinearConstraint, minimize  # NonlinearConstraint: trust-constr (r.v)

from epsilon_constraint import epsilon_constraint

# o quasi-Newton do trust-constr avisa que f1 é linear («delta_grad == 0.0»); é esperado
warnings.filterwarnings("ignore", message="delta_grad == 0.0")


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def perto(v, s, tol):
    """v está a menos de tol do valor exato s?"""
    return bool(np.all(np.abs(np.ravel(np.asarray(v, float))
                              - np.ravel(np.asarray(s, float))) <= tol))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 6.4: método do epsilon-constrangimento")

f1 = lambda x: x[0]
f2 = lambda x: 1 - x[0]**2 + x[1]
b = [(0, 1), (0, 0.6)]
x0 = [0.5, 0.3]

# ------------------------------------------------ 1. À mão: três valores
print("\n1. min f2 = 1 - x1^2 + x2  s.a.  f1 = x1 <= e  (x* = (e, 0))")
r = epsilon_constraint(f1, f2, x0, b, [0.25, 0.50, 0.75])
for Fk in r.fx:
    print("   e = %.2f  =>  (f1, f2) = (%.4f; %.4f)" % (Fk[0], Fk[0], Fk[1]))
c = confere(r.fx, [[0.25, 0.9375], [0.50, 0.75], [0.75, 0.4375]], 4)
print("  (0.25; 0.9375), (0.50; 0.75), (0.75; 0.4375): %s" % simnao([c]))
ok = ok and c

# ------------------------------------------------ 2. Varrimento do slide
print("\n2. Varrimento e = 0.05, 0.15, ..., 0.95 a partir de x0 = (%g, %g), arranque a quente" % tuple(x0))
e = np.linspace(0.05, 0.95, 10)
r = epsilon_constraint(f1, f2, x0, b, e, verbose=True)
print("   solver: %s" % r.message)
nd = len(np.unique(np.round(r.fx * 1e6), axis=0))
print("   %d valores de e -> %d pontos distintos, todos na frente f2 = 1 - f1^2" % (len(e), nd))
print("   contagens (SLSQP): %d avaliações de f2 (soma de r.nfev), %d de f1 (%d chamadas); %d iterações"
      % (r.nfev, r.nfev1, r.ncalls, r.nit))
c = [len(e) == 10, nd == 10, perto(r.x, np.c_[e, np.zeros(10)], 1e-6),
     perto(r.fx[:, 1], 1 - e**2, 1e-6), perto(r.u, 2 * e, 1e-4), r.flag == 0,
     r.nfev == 60, r.nfev1 == 90, r.ncalls == 150]
print("  10 valores: %s | 10 pontos: %s | x* = (e, 0): %s | f2 = 1 - e^2: %s | u = 2e: %s | sem falhas: %s"
      " | 60 de f2: %s | 90 de f1: %s | 150 chamadas: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 3. O multiplicador
print("\n3. Multiplicador de f1 <= e (g = e - f1 >= 0): u = 2e = -d f2*/de")
r = epsilon_constraint(f1, f2, x0, b, [0.50, 0.51, 0.90, 0.91])
d1 = r.fx[1, 1] - r.fx[0, 1]; d2 = r.fx[2, 1] - r.fx[3, 1]
print("   e = 0.50: x* = (%.4f; %.4f), f2* = %.4f, u = %.4f" % (*r.x[0], r.fx[0, 1], r.u[0]))
print("   e = 0.51: f2* = %.4f; diferença %.4f  (aproximação -u*0.01 = %.4f)"
      % (r.fx[1, 1], d1, -r.u[0] * 0.01))
print("   e = 0.90: u = %.4f; a folga 0.90 -> 0.91 compra %.4f em f2" % (r.u[2], d2))
rt = minimize(f2, x0, bounds=b, constraints=NonlinearConstraint(f1, -np.inf, 0.5), method="trust-constr")
print("   e = 0.50 com trust-constr (como no slide): u = r.v[0] = %.4f (x* = (%.4f; %.4f))"
      % (rt.v[0][0], *rt.x))
c = [confere(r.x[0, 0], 0.5, 1), confere(r.fx[0, 1], 0.75, 2), confere(r.u[0], 1, 3),
     confere(r.fx[1, 1], 0.7399, 4), confere(d1, -0.0101, 4), confere(r.u[2], 1.8, 3),
     confere(d2, 0.018, 3), confere(rt.v[0][0], 1, 2)]
print("  x* = 0.5: %s | f2* = 0.75: %s | u = 1: %s | f2(0.51) = 0.7399: %s | -0.0101: %s"
      " | u(0.9) = 1.8: %s | ~0.018: %s | trust-constr u ~ 1: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 4. Restrição inativa
print("\n4. Restrição inativa: e > 1")
r = epsilon_constraint(f1, f2, x0, b, [1.05, 1.2])
for h, xk, Fk, uk in zip(r.history, r.x, r.fx, r.u):
    print("   e = %.2f: x* = (%.4f; %.4f), f1 = %.4f < e, u = %.4f" % (h[0], *xk, Fk[0], abs(uk)))
c = perto(r.x, [[1, 0], [1, 0]], 1e-6) and perto(r.u, [0, 0], 1e-6)
print("  mesma solução (1, 0) com u = 0: %s" % simnao([c]))
ok = ok and c

# ------------------------------------------------ 5. Nantes–Lille (6.1)
print("\n5. Nantes–Lille: o mais barato com duração <= e")
nomes = ["Carro via Paris", "Carro via Rouen", "Carro sem portagens", "Autocarro 1",
         "Autocarro 2", "TGV", "TGV low-cost", "Avião"]
dur = np.array([5 + 33 / 60, 5 + 49 / 60, 7 + 18 / 60, 9.5, 9.5, 4 + 2 / 60, 4 + 20 / 60, 1 + 5 / 60])
custo = np.array([230, 251, 204, 38, 28, 84, 40, 134])
esc = []
for E in (4, 5, 10):
    adm = np.flatnonzero(dur <= E)
    j = adm[np.argmin(custo[adm])]              # empate: fica o primeiro
    esc.append(j)
    print("   e = %2d h: %s (%d €)" % (E, nomes[j], custo[j]))
c = esc == [7, 6, 4] and list(custo[esc]) == [134, 40, 28]
print("  avião (134), TGV low-cost (40), autocarro 2 (28): %s" % simnao([c]))
ok = ok and c

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 6.4: método do epsilon-constrangimento

1. min f2 = 1 - x1^2 + x2  s.a.  f1 = x1 <= e  (x* = (e, 0))
   e = 0.25  =>  (f1, f2) = (0.2500; 0.9375)
   e = 0.50  =>  (f1, f2) = (0.5000; 0.7500)
   e = 0.75  =>  (f1, f2) = (0.7500; 0.4375)
  (0.25; 0.9375), (0.50; 0.75), (0.75; 0.4375): sim

2. Varrimento e = 0.05, 0.15, ..., 0.95 a partir de x0 = (0.5, 0.3), arranque a quente
     e       x1       x2       f1       f2        u   nf2   nf1
  0.05   0.0500   0.0000   0.0500   0.9975   0.1000     6     9
  0.15   0.1500   0.0000   0.1500   0.9775   0.3000     6     9
  0.25   0.2500   0.0000   0.2500   0.9375   0.5000     6     9
  0.35   0.3500   0.0000   0.3500   0.8775   0.7000     6     9
  0.45   0.4500   0.0000   0.4500   0.7975   0.9000     6     9
  0.55   0.5500   0.0000   0.5500   0.6975   1.1000     6     9
  0.65   0.6500   0.0000   0.6500   0.5775   1.3000     6     9
  0.75   0.7500   0.0000   0.7500   0.4375   1.5000     6     9
  0.85   0.8500   0.0000   0.8

## 6.5 Direct MultiSearch (DMS)

Código de investigação do docente, distribuído na aula; não faz parte do código da UC nem deste caderno. Ver [`README.md`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/06_multiobjective_optimization/06_5_dms/README.md).

## 6.6 NSGA-II (algoritmos genéticos multiobjetivo)

Exemplo [`ex06_6_nsga2.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/06_multiobjective_optimization/06_6_nsga2/ex06_6_nsga2.py) · funções da UC: [`non_dominated_sort.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/06_multiobjective_optimization/06_6_nsga2/non_dominated_sort.py), [`crowding_distance.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/06_multiobjective_optimization/06_6_nsga2/crowding_distance.py), [`hypervolume_2d.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/06_multiobjective_optimization/06_6_nsga2/hypervolume_2d.py), [`nsga2.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/06_multiobjective_optimization/06_6_nsga2/nsga2.py)

```text
Reproduz os exemplos do deck 6.6 (NSGA-II) e o hipervolume do deck 6.2.

1. Oito soluções (frames «rank» e «crowding»): F_1 = {1,2,4,8},
   F_2 = {3,5,6,7}; d_2 = 1.4, d_4 = 1.233; só cabem 3: sai o 4.
2. Hipervolume (deck 6.2): três pontos com r = (1.1; 1.3): 0.549;
   convergência/diversidade com r = (1.1; 1.65): 0.654, 0.656, 1.095;
   frentes exatas com r = (1.1; 1.65): convexa 1.648, não convexa 1.148.
3. NSGA-II no exemplo convexo da UC, N = 40, G = 50 (semente 4):
   geração 0 com 6 não dominados, 2040 avaliações, HV 1.637.
4. Tabela «DMS vs. NSGA-II» (exemplo não convexo): coluna do NSGA-II,
   20 corridas (sementes 100-119), mediana e quartis do HV por orçamento.

Método estocástico: usa o gerador numpy.random.default_rng e as mesmas
sementes do script das figuras do capítulo 6, por isso reproduz EXATAMENTE
os números dos slides.  Os slides usam vírgula decimal; aqui usa-se o ponto.
```

In [4]:
import numpy as np

from non_dominated_sort import non_dominated_sort
from crowding_distance import crowding_distance
from hypervolume_2d import hypervolume_2d
from nsga2 import nsga2


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


def um(fr):                              # índices base 0 -> numeração do slide
    return "{" + ",".join(str(i + 1) for i in sorted(fr)) + "}"


ok = True
print("Otimização — deck 6.6: NSGA-II (e hipervolume, deck 6.2)")

# ------------------------------------------------ 1. oito soluções
Pex = np.array([[1, 6], [2, 3], [3, 5], [4, 1.5], [5, 4], [6, 2.5], [2.5, 6.5], [7, 1]])
print("\n1. Oito soluções (f1, f2): rank e crowding")
fronts, rank = non_dominated_sort(Pex)
d = crowding_distance(Pex, fronts[0])
dp = dict(zip(fronts[0], d))
print("%3s %12s %5s %9s" % ("p", "(f1; f2)", "rank", "d_p"))
for i in range(len(Pex)):
    print("%3d %12s %5d %9s" % (i + 1, "(%g; %g)" % tuple(Pex[i]), rank[i],
                                "%.3f" % dp[i] if i in dp else "--"))
print("F_1 = %s, F_2 = %s" % (um(fronts[0]), um(fronts[1])))
fica = sorted(range(len(d)), key=lambda k: -d[k])[:3]          # só cabem 3 dos 4
fica = [fronts[0][k] for k in fica]
sai = [i for i in fronts[0] if i not in fica]
print("Só cabem 3 dos 4 de F_1: ficam %s, sai %s" % (um(fica), um(sai)))
c = [fronts[0] == [0, 1, 3, 7] and sorted(fronts[1]) == [2, 4, 5, 6],
     confere(dp[1], 1.4, 1), confere(dp[3], 1.233, 3),
     bool(np.isinf(dp[0]) and np.isinf(dp[7])), sorted(fica) == [0, 1, 7] and sai == [3]]
print("  frentes: %s | d_2 = 1.4: %s | d_4 = 1.233: %s | extremos d = inf: %s | sai o 4: %s"
      % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 2. hipervolume (6.2)
print("\n2. Hipervolume (deck 6.2)")
P3 = np.array([[0.2, 0.96], [0.5, 0.75], [0.8, 0.36]])
hv3 = hypervolume_2d(P3, (1.1, 1.3))
REF = (1.1, 1.65)
ff = lambda s: np.stack([s, 1 - s**2], -1)          # frente não convexa f2 = 1 - f1^2
A = ff(np.linspace(0.40, 0.55, 10))                  # perto, concentrados
B = ff(np.linspace(0, 1, 10)) + np.array([0.12, 0.25])  # espalhados, longe
C = ff(np.linspace(0, 1, 10))                        # perto e espalhados
hvA, hvB, hvC = (hypervolume_2d(Z, REF) for Z in (A, B, C))
t = np.linspace(0, 1, 2000)
hv_cx = hypervolume_2d(np.stack([t**2, (1 - t)**2], -1), REF)   # frente convexa, 2000 pontos
hv_nc = hypervolume_2d(ff(t), REF)
hv_cx_ex = 1.1 * 1.65 - 1 / 6        # exato: r1*r2 - integral_0^1 (1 - sqrt(f1))^2 df1
hv_nc_ex = 1.1 * 1.65 - 2 / 3        # exato: r1*r2 - integral_0^1 (1 - f1^2) df1
print("três pontos, r = (1.1; 1.3): HV = %.3f" % hv3)
print("r = (1.1; 1.65): perto/concentrados %.3f, espalhados/longe %.3f, perto/espalhados %.3f"
      % (hvA, hvB, hvC))
print("frente exata convexa: %.4f (2000 pontos: %.4f); não convexa: %.4f (2000 pontos: %.4f)"
      % (hv_cx_ex, hv_cx, hv_nc_ex, hv_nc))
c = [confere(hv3, 0.549, 3), confere(hvA, 0.654, 3), confere(hvB, 0.656, 3), confere(hvC, 1.095, 3),
     confere([hv_cx_ex, hv_cx], 1.648, 3), confere([hv_nc_ex, hv_nc], 1.148, 3)]
print("  0.549: %s | 0.654: %s | 0.656: %s | 1.095: %s | 1.648: %s | 1.148: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 3. NSGA-II, exemplo convexo
f = lambda x: np.array([x[0]**2 + x[1]**2, (x[0] - 1)**2 + x[1]**2])
cv = lambda x: max(0.0, x[0]**2 + x[1]**2 - 1.0)    # x1^2 + x2^2 <= 1 (regra de Deb)
print("\n3. NSGA-II no exemplo convexo: N = 40, G = 50, semente 4, r = (1.1; 1.65)")
r = nsga2(f, [0, 0], [1, 1], 40, 50, 4, cv=cv, ref=REF, verbose=True)
H = r.history
print("geração 0: %d não dominados (%d inadmissíveis); geração 50: %d pontos, "
      "n_f = %d, HV = %.4f (exata %.4f)" % (H[0, 2], H[0, 3], len(r.F), r.nfev, H[-1, 4], hv_cx_ex))
print(r.message)
c = [H[0, 2] == 6, r.nfev == 2040, confere(H[-1, 4], 1.637, 3)]
print("  6 não dominados na geração 0: %s | n_f = 2040: %s | HV = 1.637: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ 4. DMS vs. NSGA-II
fn = lambda x: np.array([x[0], 1 - x[0]**2 + x[1]])  # exemplo não convexo, x em [0,1]x[0,0.6]
print("\n4. DMS vs. NSGA-II no exemplo não convexo (r = (1.1; 1.65)), coluna do NSGA-II")
r = nsga2(fn, [0, 0], [1, 0.6], 40, 50, 5, ref=REF)
print("uma corrida (semente 5, G = 50): %d pontos, n_f = %d, HV = %.4f"
      % (len(r.F), r.nfev, r.history[-1, 4]))
c = [len(r.F) == 40, r.nfev == 2040]
BUDGET = 2000
HV = []
for s in range(100, 120):                            # 20 corridas, G = 2000/40 - 1 = 49
    rs = nsga2(fn, [0, 0], [1, 0.6], 40, BUDGET // 40 - 1, s, ref=REF)
    HV.append(rs.history[:, 4])
ev = rs.history[:, 1]                                # n_f de cada geração: 40, 80, ..., 2000
q1, med, q3 = np.percentile(np.array(HV), [25, 50, 75], axis=0)
orc = [100, 200, 400, 800, 2000]
dms = [1.064, 1.094, 1.123, 1.139, 1.144]            # coluna do DMS (slide; ver 06_5_dms)
Tn = [[0.986, 0.968, 1.006], [1.083, 1.066, 1.096], [1.125, 1.117, 1.129],
      [1.132, 1.131, 1.133], [1.132, 1.132, 1.133]]  # slide: mediana [Q1; Q3]
print("%8s %8s %10s %18s %24s" % ("n_f", "DMS", "NSGA med.", "[Q1; Q3]", "slide: med. [Q1; Q3]"))
obt = []
for e, dm, T in zip(orc, dms, Tn):
    k = np.searchsorted(ev, e, side="right") - 1     # última geração completa que não excede
    obt.append([med[k], q1[k], q3[k]])
    print("%8d %8.3f %10.3f   [%.3f; %.3f] %12.3f [%.3f; %.3f]" % (e, dm, med[k], q1[k], q3[k], *T))
c.append(confere(obt, Tn, 3))
print("  uma corrida: 40 pontos: %s | 2040 aval.: %s | tabela NSGA-II (20 corridas): %s" % simnao(c))
ok = ok and all(c)

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 6.6: NSGA-II (e hipervolume, deck 6.2)

1. Oito soluções (f1, f2): rank e crowding
  p     (f1; f2)  rank       d_p
  1       (1; 6)     1       inf
  2       (2; 3)     1     1.400
  3       (3; 5)     2        --
  4     (4; 1.5)     1     1.233
  5       (5; 4)     2        --
  6     (6; 2.5)     2        --
  7   (2.5; 6.5)     2        --
  8       (7; 1)     1       inf
F_1 = {1,2,4,8}, F_2 = {3,5,6,7}
Só cabem 3 dos 4 de F_1: ficam {1,2,8}, sai {4}
  frentes: sim | d_2 = 1.4: sim | d_4 = 1.233: sim | extremos d = inf: sim | sai o 4: sim

2. Hipervolume (deck 6.2)


três pontos, r = (1.1; 1.3): HV = 0.549
r = (1.1; 1.65): perto/concentrados 0.654, espalhados/longe 0.656, perto/espalhados 1.095
frente exata convexa: 1.6483 (2000 pontos: 1.6482); não convexa: 1.1483 (2000 pontos: 1.1481)
  0.549: sim | 0.654: sim | 0.656: sim | 1.095: sim | 1.648: sim | 1.148: sim

3. NSGA-II no exemplo convexo: N = 40, G = 50, semente 4, r = (1.1; 1.65)


    t     n_f   |F_1|   inadm        HV
    0      40       6      14    1.5051
    5     240      40       0    1.6126
   10     440      40       0    1.6360
   15     640      40       0    1.6359
   20     840      40       0    1.6367
   25    1040      40       0    1.6369
   30    1240      40       0    1.6345
   35    1440      40       0    1.6354
   40    1640      40       0    1.6366
   45    1840      40       0    1.6362
   50    2040      40       0    1.6369
geração 0: 6 não dominados (14 inadmissíveis); geração 50: 40 pontos, n_f = 2040, HV = 1.6369 (exata 1.6483)
fez as 50 gerações pedidas: 40 não dominados em P_G
  6 não dominados na geração 0: sim | n_f = 2040: sim | HV = 1.637: sim

4. DMS vs. NSGA-II no exemplo não convexo (r = (1.1; 1.65)), coluna do NSGA-II


uma corrida (semente 5, G = 50): 40 pontos, n_f = 2040, HV = 1.1332


     n_f      DMS  NSGA med.           [Q1; Q3]     slide: med. [Q1; Q3]
     100    1.064      0.986   [0.968; 1.006]        0.986 [0.968; 1.006]
     200    1.094      1.083   [1.066; 1.096]        1.083 [1.066; 1.096]
     400    1.123      1.125   [1.117; 1.129]        1.125 [1.117; 1.129]
     800    1.139      1.132   [1.131; 1.133]        1.132 [1.131; 1.133]
    2000    1.144      1.132   [1.132; 1.133]        1.132 [1.132; 1.133]
  uma corrida: 40 pontos: sim | 2040 aval.: sim | tabela NSGA-II (20 corridas): sim

confere com os slides: sim
